# Autonomous Car Racing AI using Reinforcement Learning
## Comprehensive End-to-End Implementation, Training, and Evaluation on Gymnasium CarRacing-v3

### Abstract
This research notebook presents an autonomous driving agent trained to navigate complex racing tracks in **Gymnasium CarRacing-v3** using Deep Reinforcement Learning. We implement, train, and compare two distinct algorithmic paradigms:
1. **Continuous Proximal Policy Optimization (PPO)**: Multimodal continuous steering, throttle, and braking with a Convolutional Neural Network (CNN).
2. **Discrete Deep Q-Network (DQN)**: Action-value approximation over a 7-action driving maneuver palette.
3. **Random Continuous Baseline**: Stochastic uniform action sampling benchmark.

In [ ]:
import os
import sys
sys.path.append('..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gymnasium as gym

from environments.car_racing_env import make_car_racing_env
from environments.discrete_car_racing_env import make_discrete_car_racing_env, DISCRETE_ACTIONS
from agents.agent_factory import load_racing_agent
from evaluation.metrics import compute_racing_metrics, compare_racing_distributions

sns.set_theme(style="whitegrid")
print("Autonomous Car Racing AI environment modules imported successfully!")

### 1. Environment Observation Space & Action Spaces
CarRacing-v3 presents a continuous 2D top-down driving simulation with Box2D physics. Raw RGB observations are `(96, 96, 3)`. We apply grayscale conversion and downsampling to `(84, 84, 1)` with `FrameSkip(2)` for CPU efficiency.

In [ ]:
# Continuous CarRacing-v3 Environment for PPO
env_cont = make_car_racing_env(continuous=True, seed=42)
obs_cont, info_cont = env_cont.reset(seed=42)
print(f"Continuous Observation Space: {env_cont.observation_space}")
print(f"Continuous Action Space:      {env_cont.action_space}")
print(f"Initial Telemetry:             {info_cont.get('telemetry', {})}")
env_cont.close()

# Discrete CarRacing-v3 Environment for DQN
env_disc = make_discrete_car_racing_env(seed=42)
obs_disc, info_disc = env_disc.reset(seed=42)
print(f"\nDiscrete Observation Space:   {env_disc.observation_space}")
print(f"Discrete Action Space:        {env_disc.action_space} (7 discrete maneuvers)")
for idx, act in DISCRETE_ACTIONS.items():
    print(f"  Action {idx}: Steer={act[0]:+.1f}, Gas={act[1]:.1f}, Brake={act[2]:.1f}")
env_disc.close()

### 2. Multi-Agent Benchmark Evaluation Results
We evaluate trained agents across identical procedural track seeds, measuring episodic returns, track progress %, and off-track events.

In [ ]:
comp_csv = '../results/evaluation/racing_algorithm_comparison.csv'
if os.path.exists(comp_csv):
    comp_df = pd.read_csv(comp_csv)
    summary = comp_df.groupby('algorithm').agg({
        'reward': ['mean', 'std', 'median', 'max'],
        'track_progress_percent': ['mean', 'max'],
        'length': ['mean']
    }).round(2)
    display(summary)
else:
    print("Run evaluation/compare_agents.py to generate comparative dataset.")

### 3. Empirical Return & Track Progress Distributions
Visualizing distribution of cumulative episodic returns and track completion across agents.

In [ ]:
if os.path.exists(comp_csv):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    sns.boxplot(data=comp_df, x='algorithm', y='reward', ax=ax1, palette='Set2')
    sns.stripplot(data=comp_df, x='algorithm', y='reward', color='black', alpha=0.3, jitter=0.2, ax=ax1)
    ax1.set_title('Episodic Return Distribution', fontweight='bold')
    ax1.axhline(0, color='gray', linestyle=':')
    
    if 'track_progress_percent' in comp_df.columns:
        sns.barplot(data=comp_df, x='algorithm', y='track_progress_percent', ax=ax2, palette='Set2', ci='sd')
        ax2.set_title('Mean Track Progress (%)', fontweight='bold')
        ax2.set_ylim(0, 100)
    
    plt.tight_layout()
    plt.show()

### 4. Statistical Significance Testing
We apply Welch's two-sample t-test (unequal variances) and Mann-Whitney U test to verify that continuous control statistically outperforms discrete control and random exploration.

In [ ]:
comp_json = '../results/evaluation/racing_algorithm_comparison.json'
if os.path.exists(comp_json):
    import json
    with open(comp_json, 'r') as f:
        data = json.load(f)
    stat_tests = data.get('statistical_tests', {})
    print("=== Statistical Hypothesis Tests ===")
    for k, v in stat_tests.items():
        print(f"\n{k.upper()}:")
        print(f"  Winner:             {v.get('winner')}")
        print(f"  Welch t-test p-val: {v.get('p_value_welch_t'):.4f}")
        print(f"  Mann-Whitney p-val: {v.get('p_value_mann_whitney'):.4f}")
        print(f"  Cohen's d:          {v.get('cohens_d'):.2f}")
        print(f"  Significant (p<.05): {v.get('is_significant_at_05')}")